# 01 — Cleaning
Load the raw survey export, apply quality filters, and produce `data/clean.csv`.

Steps: attention check → GPA validation → range answers to numeric → one-hot the multi-select.

In [ ]:
import pandas as pd

raw = pd.read_csv('../data/responses.csv')
raw.columns = [c.strip() for c in raw.columns]
print(f"{len(raw)} raw responses")
raw.head(3)

In [ ]:
COLUMNS = {
    'GPA (0.0–4.0)': 'gpa',
    'Hours studied per week': 'study_hours',
    'Study consistency (1–5)': 'consistency',
    'What is your primary study method?': 'method',
    'How often do you use AI tools (ChatGPT, etc.) for studying?': 'ai_freq',
    'How do you primarily use AI? (Select all that apply)': 'ai_uses',
    'How confident are you in using AI effectively?': 'ai_confidence',
    'How dependent are you on AI for completing assignments?': 'ai_dependence',
    'How often do you verify or double-check AI-generated answers?': 'ai_verify',
    'What is your major?': 'major',
    'What is your current year in school?': 'year',
    'To ensure quality responses, please select \u2018Often\u2019 for this question': 'attention',
    'How difficult are your current courses?': 'difficulty',
    'On average, how many hours do you sleep per night? (Number)': 'sleep',
    'How easily do you get distracted while studying?': 'distraction',
    'How often do you complete assignments without external help (including AI)?': 'independence',
}
df = raw.rename(columns=COLUMNS)

In [ ]:
# Attention check: keep only rows that selected 'Often'
before = len(df)
df = df[df['attention'].astype(str).str.strip().str.lower() == 'often']
print(f"Attention check: dropped {before - len(df)}")

# GPA: numeric, within 0.0-4.0
df['gpa'] = pd.to_numeric(df['gpa'], errors='coerce')
before = len(df)
df = df[df['gpa'].between(0.0, 4.0)]
print(f"GPA validity: dropped {before - len(df)}")

In [ ]:
# Range answers -> numeric midpoints
HOURS = {'0-5 Hours': 2.5, '5\u201310 Hours': 7.5, '10\u201320 Hours': 15,
         '20\u201330 Hours': 25, '30+ Hours': 35}
SLEEP = {'<5': 4.5, '5\u20136': 5.5, '6\u20137': 6.5, '7\u20138': 7.5, '8+': 8.5}
df['study_hours_n'] = df['study_hours'].map(HOURS)
df['sleep_n'] = df['sleep'].map(SLEEP)

for col in ['consistency', 'ai_freq', 'ai_confidence', 'ai_dependence',
            'ai_verify', 'difficulty', 'distraction', 'independence']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# One-hot encode the multi-select 'How do you use AI'
uses = df['ai_uses'].fillna('').str.get_dummies(sep=', ')
uses.columns = ['use_' + c.strip().lower().replace(' ', '_').replace('/', '')[:25]
                for c in uses.columns]
df = pd.concat([df, uses], axis=1)

df = df.drop(columns=['attention', 'Timestamp'], errors='ignore')
df.to_csv('../data/clean.csv', index=False)
print(f"Saved {len(df)} clean responses -> data/clean.csv")
df.head(3)